# Official HEC-HMS Guide Mirror: GIS Tools and Terrain Data

Official guide: https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/gis-tools-and-terrain-data

This notebook mirrors the official GIS guide category with hms-commander operations for model-type detection, GIS asset inventory, basin-element parsing, and GeoJSON export. It does not launch the HMS GUI GIS tools; it demonstrates the programmatic bridge available today.

In [1]:
from pathlib import Path
import logging

import pandas as pd

logging.disable(logging.CRITICAL)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "hms_commander").is_dir() and (candidate / "examples").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
WORK_ROOT = REPO_ROOT / "examples" / "working" / "clb238_guides"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
HMS_VERSION = "4.13"


from hms_commander import HmsExamples, HmsPrj

available_versions = HmsExamples.list_versions()
if HMS_VERSION not in available_versions:
    HMS_VERSION = available_versions[0]
HMS_EXE = HmsExamples.get_hms_exe(HMS_VERSION)


def init_sample_project(project_name, notebook_key):
    project_path = HmsExamples.extract_project(
        project_name,
        version=HMS_VERSION,
        output_path=WORK_ROOT / notebook_key,
        overwrite=True,
    )
    project = HmsPrj()
    project.initialize(project_path, hms_exe_path=HMS_EXE)
    return project, project_path

In [2]:
from hms_commander import HmsBasin, HmsGeo

castro, castro_path = init_sample_project("castro", "24_gis_terrain_data")
river_bend, river_bend_path = init_sample_project("river_bend", "24_gis_terrain_data")

model_types = pd.DataFrame([
    {"sample_project": "castro", "model_type": HmsGeo.detect_model_type(castro_path)},
    {"sample_project": "river_bend", "model_type": HmsGeo.detect_model_type(river_bend_path)},
])
assert set(model_types["model_type"]).issubset({"lumped", "gridded"})
model_types

,sample_project,model_type
0,castro,lumped
1,river_bend,gridded


In [3]:
map_assets = []
for sample_name, sample_path in [("castro", castro_path), ("river_bend", river_bend_path)]:
    maps_dir = sample_path / "maps"
    for suffix in [".shp", ".dbf", ".prj", ".shx", ".gdr"]:
        map_assets.append({
            "sample_project": sample_name,
            "suffix": suffix,
            "count": len(list(maps_dir.glob(f"*{suffix}"))) if maps_dir.exists() else 0,
        })
asset_summary = pd.DataFrame(map_assets)
assert asset_summary["count"].sum() > 0
asset_summary

,sample_project,suffix,count
0,castro,.shp,2
1,castro,.dbf,2
2,castro,.prj,2
3,castro,.shx,2
4,castro,.gdr,2
5,river_bend,.shp,2
6,river_bend,.dbf,2
7,river_bend,.prj,2
8,river_bend,.shx,2
9,river_bend,.gdr,2


In [4]:
basin_path = Path(castro.basin_df.iloc[0]["full_path"])
subbasins, junctions, reaches = HmsGeo.parse_basin_file(basin_path)
element_summary = pd.DataFrame([
    {"element_type": "subbasins", "count": len(subbasins)},
    {"element_type": "junctions", "count": len(junctions)},
    {"element_type": "reaches", "count": len(reaches)},
])
assert element_summary["count"].sum() > 0
element_summary

,element_type,count
0,subbasins,4
1,junctions,3
2,reaches,2


In [5]:
gis_output_dir = WORK_ROOT / "24_gis_terrain_data" / "geojson"
gis_outputs = HmsGeo.extract_all_gis(
    basin_path,
    output_dir=gis_output_dir,
    crs_epsg="EPSG:4326",
)
output_summary = pd.DataFrame([
    {"layer": layer, "file": path.name, "exists": path.exists(), "size_kb": round(path.stat().st_size / 1024, 1)}
    for layer, path in sorted(gis_outputs.items())
])
assert output_summary["exists"].all()
output_summary

,layer,file,exists,size_kb
0,junctions,hms_junctions.geojson,True,1.1
1,reaches,hms_reaches.geojson,True,1.1
2,subbasins,hms_subbasins.geojson,True,1.6


## Coverage Notes

The official GIS category also covers georeferencing, terrain preprocessing, linking terrain data, delineation, and HMS GIS parameter estimation. The existing hms-commander surface covers GIS extraction and TauDEM-oriented workflows; soil/land-use parameter-estimation parity is tracked in CLB-289.